# Deploy Multi-Agent Financial Advisor on Amazon Bedrock AgentCore Runtime

This notebook deploys the multi-agent financial advisory system to **Amazon Bedrock AgentCore Runtime** — a serverless, purpose-built hosting environment for AI agents.

## What This Lab Does

- Deploys the **orchestrator agent** (Claude Haiku 4.5 via Bedrock) and **financial analysis agent** (Qwen 3.5 9B via SageMaker OpenAI-compatible API) as a containerized service on AgentCore Runtime
- Uses **IAM SigV4** authentication — invoke via `boto3` with standard AWS credentials
- Includes full observability (CloudWatch Logs, X-Ray traces, token usage metrics)

## Prerequisites

- Qwen 3.5 9B endpoint deployed on SageMaker (from Lab 1)
- IAM role with AgentCore and SageMaker permissions
- Amazon Bedrock model access for Claude Haiku 4.5

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts bedrock-agentcore-starter-toolkit boto3 strands-agents python-dotenv

In [ ]:
import uuid
import json
import os
import boto3
from pathlib import Path
from boto3.session import Session
from dotenv import load_dotenv
from bedrock_agentcore_starter_toolkit import Runtime

load_dotenv(Path("..") / ".env", override=True)

boto_session = Session()
region = os.environ["AWS_REGION"]
print(f"Region: {region}")

## Step 1: Prepare Agent Code for AgentCore Runtime

AgentCore requires:
- A `main.py` with `BedrockAgentCoreApp` + `@app.entrypoint` decorator
- A `requirements.txt` for container dependencies
- A `budget_agent.py` module (copied from Lab2)

In [ ]:
# Copy budget_agent.py from Lab2/strands_agents into this folder for container packaging
import shutil
from pathlib import Path

src = Path("..") / "Lab2" / "strands_agents" / "budget_agent.py"
dst = Path(".") / "budget_agent.py"
shutil.copy2(src, dst)
print(f"✓ Copied {src} → {dst}")

In [ ]:
%%writefile main.py
# See Lab3/main.py for the full AgentCore Runtime entrypoint code
# This cell writes the main.py file used by AgentCore
# The full implementation is already in this directory

In [ ]:
%%writefile requirements.txt
strands-agents[otel]>=1.0.0
boto3
botocore
openai
httpx
yfinance
pydantic
bedrock-agentcore
sagemaker-core
python-dotenv

## Step 2: Test Locally

Before deploying to AgentCore, verify the agent server works locally:

**Terminal 1** — Start the server:
```bash
python main.py
```

**Terminal 2** — Send a test request:
```bash
curl -X POST http://localhost:8080/invocations \\
   -H "Content-Type: application/json" \\
   -d '{"prompt": "Analyze Apple stock"}'
```

## Step 3: Configure and Deploy to AgentCore Runtime

In [ ]:
# Configure AgentCore Runtime
agentcore_runtime = Runtime()
agent_name = "personal_finance_agent"

print("Configuring AgentCore Runtime (IAM SigV4 auth)...")
response = agentcore_runtime.configure(
    entrypoint="main.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
)
print("Configuration completed ✓")

### Important: Add SageMaker permissions to the execution role

The auto-created execution role needs permission to invoke the SageMaker endpoint.

Permissions added:
- `sagemaker:InvokeEndpoint` — standard inference
- `sagemaker:InvokeEndpointWithResponseStream` — streaming inference
- `sagemaker:CallWithBearerToken` — required for OpenAI-compatible API path

In [ ]:
import json
import boto3

iam = boto3.client("iam")
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]

# Find the auto-created AgentCore execution role
paginator = iam.get_paginator("list_roles")
execution_role_name = None
for page in paginator.paginate(PathPrefix="/"):
    for role in page["Roles"]:
        if role["RoleName"].startswith(f"AmazonBedrockAgentCoreSDKRuntime-{region}"):
            execution_role_name = role["RoleName"]
            break
    if execution_role_name:
        break

if not execution_role_name:
    print("❌ Could not find AgentCore execution role. Run the configure() cell first.")
else:
    sagemaker_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {"Sid": "SageMakerEndpointInvoke", "Effect": "Allow",
             "Action": ["sagemaker:InvokeEndpoint", "sagemaker:InvokeEndpointWithResponseStream"],
             "Resource": f"arn:aws:sagemaker:{region}:{account_id}:endpoint/qwen35-9b-*"},
            {"Sid": "SageMakerBearerToken", "Effect": "Allow",
             "Action": "sagemaker:CallWithBearerToken",
             "Resource": f"arn:aws:sagemaker:{region}:{account_id}:endpoint/qwen35-9b-*"}
        ]
    }
    iam.put_role_policy(RoleName=execution_role_name, PolicyName="SageMakerEndpointAccess", PolicyDocument=json.dumps(sagemaker_policy))
    print(f"✓ Attached SageMaker permissions to role: {execution_role_name}")

In [ ]:
# Launch the agent to AgentCore Runtime
print("Launching agent to AgentCore Runtime...")
print("This may take several minutes (builds container, pushes to ECR, creates runtime)...")

launch_result = agentcore_runtime.launch(
    env_vars={
        "SAGEMAKER_ENDPOINT_NAME": os.environ["SAGEMAKER_ENDPOINT_NAME"],
        "AWS_REGION": os.environ["AWS_REGION"],
        "OTEL_PYTHON_EXCLUDED_URLS": "/ping,/invocations",
    }
)

print(f"\n✓ Launch completed!")
print(f"  Agent ARN: {launch_result.agent_arn}")
print(f"  Agent ID:  {launch_result.agent_id}")
print(f"  ECR URI:   {launch_result.ecr_uri}")

## Step 4: Invoke the Agent

We invoke via the `bedrock-agentcore` boto3 client with IAM SigV4 signing.

In [ ]:
# Invoke using boto3 with IAM SigV4
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

session_id = str(uuid.uuid4())
# Replace with your actual agent runtime ARN after deployment
agentRuntimeArn = launch_result.agent_arn

response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agentRuntimeArn,
    payload=json.dumps({
        "prompt": "I make $6000/month and want to invest $500/month. Help me create a budget and suggest a portfolio."
    }).encode("utf-8"),
    runtimeSessionId=session_id,
)

# Process streaming response
print("Agent Response:")
print("=" * 60)
content_type = response.get("contentType", "")
if "text/event-stream" in content_type:
    for line in response["response"].iter_lines(chunk_size=10):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                print(line[6:], end="", flush=True)
    print()
print("=" * 60)

## Summary

### SageMaker OpenAI Endpoint on AgentCore — Key Points

1. The `generate_token()` function works inside AgentCore containers because the execution role provides AWS credentials
2. The execution role must have `sagemaker:InvokeEndpoint` and `sagemaker:CallWithBearerToken` permissions
3. The `httpx.AsyncClient` + `SageMakerAuth` pattern refreshes tokens automatically
4. Invocation uses IAM SigV4 signing via `boto3` — standard AWS credential-based access

## Cleanup (Optional)

In [ ]:
# Uncomment to delete AgentCore Runtime and ECR repository

# agentcore_control = boto3.client("bedrock-agentcore-control", region_name=region)
# ecr_client = boto3.client("ecr", region_name=region)

# agentcore_control.delete_agent_runtime(agentRuntimeId=launch_result.agent_id)
# print(f"✓ Runtime '{launch_result.agent_id}' deleted")

# ecr_repo_name = launch_result.ecr_uri.split("/")[1].split(":")[0]
# ecr_client.delete_repository(repositoryName=ecr_repo_name, force=True)
# print(f"✓ ECR repo '{ecr_repo_name}' deleted")